### OpenAI API Programming
### Identifying website links and Create a Company Brochure for a website. 
### Ex: TCS and Huggingface websites

In [14]:
import os
import requests
import getpass
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API KEY:")

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

MODEL = 'gpt-4o-mini'
#MODEL = 'gpt-4o'
openai = OpenAI()

In [16]:
# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"


In [17]:
tcs = Website("https://tcs.com")
tcs.links


['#main-content',
 '#baseFooter',
 'https://www.tcs.com',
 '#',
 'https://www.tcs.com/contact-us/whats-on-your-mind',
 'https://www.tata.com/',
 'https://www.tcs.com/what-we-do/perpetually-adaptive-enterprise',
 'https://www.tcs.com/what-we-do.html#industries',
 'https://www.tcs.com/what-we-do.html#services',
 'https://www.tcs.com/what-we-do.html#products',
 'https://www.tcs.com/what-we-do/research',
 'https://www.tcs.com/who-we-are/alliances-partnerships',
 'https://www.tcs.com/what-we-do/industries/banking',
 'https://www.tcs.com/what-we-do/industries/capital-markets',
 'https://www.tcs.com/what-we-do/industries/consumer-packaged-goods-and-distribution',
 'https://www.tcs.com/what-we-do/industries/communications-media-information-services',
 'https://www.tcs.com/what-we-do/industries/education',
 'https://www.tcs.com/what-we-do/industries/energy-resources-utilities',
 'https://www.tcs.com/what-we-do/industries/healthcare',
 'https://www.tcs.com/what-we-do/industries/high-tech',
 'htt

In [18]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}
"""

In [19]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}



In [20]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [22]:
print(get_links_user_prompt(tcs))

Here is the list of links on the website of https://tcs.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
#main-content
#baseFooter
https://www.tcs.com
#
https://www.tcs.com/contact-us/whats-on-your-mind
https://www.tata.com/
https://www.tcs.com/what-we-do/perpetually-adaptive-enterprise
https://www.tcs.com/what-we-do.html#industries
https://www.tcs.com/what-we-do.html#services
https://www.tcs.com/what-we-do.html#products
https://www.tcs.com/what-we-do/research
https://www.tcs.com/who-we-are/alliances-partnerships
https://www.tcs.com/what-we-do/industries/banking
https://www.tcs.com/what-we-do/industries/capital-markets
https://www.tcs.com/what-we-do/industries/consumer-packaged-goods-and-distribution
https://www.tcs.com/what-we-do/industries/communications-media-information-services
https://www.tcs.com/wh

In [23]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [24]:
huggingface = Website("https://huggingface.co")
huggingface.links


['/',
 '/models',
 '/datasets',
 '/spaces',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/Qwen/Qwen3.5-9B',
 '/Qwen/Qwen3.5-35B-A3B',
 '/Lightricks/LTX-2.3',
 '/Qwen/Qwen3.5-0.8B',
 '/Qwen/Qwen3.5-4B',
 '/models',
 '/spaces/FrameAI4687/Omni-Video-Factory',
 '/spaces/pliny-the-prompter/obliteratus',
 '/spaces/r3gm/wan2-2-fp8da-aoti-preview',
 '/spaces/prithivMLmods/FireRed-Image-Edit-1.0-Fast',
 '/spaces/multimodalart/qwen-image-multiple-angles-3d-camera',
 '/spaces',
 '/datasets/TuringEnterprises/Open-RL',
 '/datasets/nohurry/Opus-4.6-Reasoning-3000x-filtered',
 '/datasets/crownelius/Opus-4.6-Reasoning-3300x',
 '/datasets/togethercomputer/CoderForge-Preview',
 '/datasets/LeeXiangNO1/DyNativeGaussian_sequence',
 '/datasets',
 '/join',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/inference/models',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/allenai',
 '/facebook'

In [25]:
get_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'community discussion page',
   'url': 'https://discuss.huggingface.co'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

In [26]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [27]:
print(get_all_details("https://huggingface.co"))

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'contact page', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}
Landing page:
Webpage Title:
Hugging Face – The AI community building the future.
Webpage Contents:
Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
B

In [28]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

In [29]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [30]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'company page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}


'You are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHugging Face – The AI community building the future.\nWebpage Contents:\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-9B\nUpdated\n6 days ago\n•\n693k\n•\n570\nQwen/Qwen3.5-35B-A3B\nUpdated\n9 days ago\n•\n1.09M\n•\n1.03k\nLightricks/LTX-2.3\nUpdated\n2 days ago\n•\n119k\n•\n323\nQwen/Qwen3.5-0.8B\nUpdated\n6 days ago\n•\n346k\n•\n315\nQwen/Qwen3.5-4B\nUpdated\n6 days ago\n•\n283k\n•\n285\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n418\nOmni Video Factory\n🏆\n418\ntext to video, 

In [31]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [32]:
create_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'community discussion page', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}]}


```markdown
# Hugging Face: Empowering the AI Community

## Who We Are
Hugging Face is at the forefront of artificial intelligence, serving as a collaborative platform for the machine learning community. Our mission is simple: to build the future of AI by enabling individuals and organizations to easily create, share, and harness machine learning models, datasets, and applications.

## Our Offerings
- **Models**: Home to over 2 million models, Hugging Face provides a library that is continuously updated, allowing users to explore trending solutions and cutting-edge technology.
- **Datasets**: With access to 500,000 datasets, we help developers find the data they need to fuel their projects.
- **Spaces**: Experience a suite of applications powered by AI, including features for text-to-video generation and image editing.
- **Enterprise Solutions**: Customized offerings for organizations looking to utilize AI at scale, featuring solutions for priority support, single sign-on, and more.

## Company Culture
At Hugging Face, we pride ourselves on fostering an open and collaborative environment. Our team is comprised of innovators who believe in the power of community-driven development and the importance of open-source contributions in the AI space. We encourage creativity and an entrepreneurial spirit, empowering team members to explore new ideas while working towards shared goals.

## Our Customers
We serve a diverse range of customers, from startups to Fortune 500 companies. More than 50,000 organizations utilize our platform, including giants like Google, Microsoft, Amazon, and Meta. Whether you’re an enterprise looking for advanced tech or an independent developer seeking innovative solutions, Hugging Face adapts to meet your needs.

## Careers at Hugging Face
Join a passionate and talented team dedicated to shaping the future of AI! We're always on the lookout for skilled individuals who are eager to contribute to an innovative and inclusive culture. With a variety of roles available, from engineering to marketing, there’s an opportunity for everyone to thrive.

### Current Openings:
- Software Engineers
- Data Scientists
- Product Managers
- Community Managers

Explore your future at Hugging Face and be a part of the AI revolution!

## Get Involved
Whether you're looking to adopt AI into your projects, collaborate with a vibrant community, or join our team, Hugging Face welcomes you to be a part of this incredible journey. Sign up on our website today to start exploring!

### Connect with Us
- [Hugging Face Website](https://huggingface.co/)
- [Twitter](https://twitter.com/huggingface)
- [LinkedIn](https://www.linkedin.com/company/huggingface)
- [Discord](https://discord.gg/huggingface)

Join us as we shape the future of AI, together!

---
*This brochure is for informational purposes only and is not intended as a complete representation of Hugging Face.*
```

In [33]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [34]:
stream_brochure("TCS", "https://tcs.com")

Found links: {'links': [{'type': 'about page', 'url': 'https://www.tcs.com/who-we-are'}, {'type': 'careers page', 'url': 'https://www.tcs.com/careers'}, {'type': 'what we do page', 'url': 'https://www.tcs.com/what-we-do.html'}, {'type': 'corporate social responsibility page', 'url': 'https://www.tcs.com/who-we-are/corporate-social-responsibility'}, {'type': 'leadership page', 'url': 'https://www.tcs.com/who-we-are/leadership'}, {'type': 'our aspiration page', 'url': 'https://www.tcs.com/who-we-are/our-aspiration'}, {'type': 'events page', 'url': 'https://www.tcs.com/who-we-are/events'}, {'type': 'newsroom page', 'url': 'https://www.tcs.com/who-we-are/newsroom'}, {'type': 'sustainable business page', 'url': 'https://www.tcs.com/who-we-are/tcs-sustainable-business-carbon-neutrality'}, {'type': 'diversity equity inclusion page', 'url': 'https://www.tcs.com/who-we-are/diversity-equity-inclusion'}]}



# TCS Brochure

## Welcome to Tata Consultancy Services (TCS)

### Building Perpetually Adaptive Enterprises

At TCS, we empower organizations to become perpetually adaptive, enabling them to thrive in a world of rapid change. Our expertise transcends mere consultancy; we deliver tailored solutions that transform complexities into opportunities, driving meaningful change for our clients and communities.

---

### Who We Are

TCS is a global leader in IT services, consulting, and business solutions, recognized for our commitment to excellence. We harness the best talent and the latest technologies which help businesses navigate their digital transformation journeys effectively.

---

### Our Services

We offer a wide array of services across various industries. Our key service areas include:

- **Artificial Intelligence and Data Analytics**
- **Cloud Solutions**
- **Cognitive Business Operations**
- **Cybersecurity**
- **Enterprise Solutions**

### Industries We Serve

TCS works across multiple sectors, ensuring tailored services to meet specific needs. Key industries include:

- Banking & Financial Services
- Healthcare
- Energy & Utilities
- Retail
- Manufacturing
- Travel & Logistics
- High Tech

---

### Our Commitment to Innovation

Innovation is at the core of our business. Through TCS Research and our advanced platforms like Quartz™, ignio™, and TCS iON™, we drive research and transformative solutions that empower enterprises to adapt continuously.

---

### Our Company Culture

At TCS, we celebrate diversity and believe in nurturing talents from all walks of life. Our inclusive environment fosters collaboration, ensuring that unique perspectives are valued. We are committed to sustainability and community development because we believe in making a positive impact on society.

---

### Careers at TCS

Do you aspire to be a global change-maker? Join our team! We are recruiting passionate individuals who are enthusiastic about driving innovation and contributing to transformative projects. 

- **Why Join Us?**
  - Opportunities to work on groundbreaking technologies
  - A supportive and dynamic work environment
  - Commitment to professional development and growth

If you're ready to make a difference, check our careers page for current openings across the globe.

---

### Join Us on Our Journey

TCS is not just a company; it’s a commitment to creating exceptional value for our customers and communities every day. Explore more about our services, industries, and career opportunities by visiting [TCS Official Website](https://www.tcs.com).

---

**Contact Us**
For inquiries or more information, please reach out through [Contact Us](https://www.tcs.com/contactus). 

